# D6 Training Pipelines — Block-Diagram Reference

**Scope.** This notebook diagrams the three D6 paper-arm trainers:

| Arm | File | Role |
|---|---|---|
| `baseline` | `src/co_scientist/d6_grant_proposal/train_baseline.py` | Frozen Qwen3-30B-A3B + oracle, **no training** — lower bound. |
| `GRPO` | `src/co_scientist/d6_grant_proposal/train_grpo.py` | Reward-only ablation — group-relative PPO on Qwen 12-signal grader. |
| `OPD` | `src/co_scientist/d6_grant_proposal/train_opd.py` | **Paper main arm** — privileged-Opus reviewer + teacher–student logprob distillation. |

The deferred trainers (`train_opd_grounded`, `train_opd_kl_anchor`) are
**out of scope** per `projects/d6_grant_proposal/STATUS.md`.

**How to read this notebook.** Each per-trainer section has the same
sub-structure: pre-loop setup flowchart → per-iteration flowchart →
prompt inventory → data flow → loss/advantage code excerpt →
doc-vs-code consistency note. Every concrete claim is anchored to
`file:line`. Cell §6 is a side-by-side cross-arm view that makes the
OPD-only components visually obvious.

**Rendering.** Mermaid blocks render natively in JupyterLab 4.x and on
GitHub. If you open this notebook somewhere that does not render
Mermaid, the source survives as fenced code in the markdown.

**Source extraction date.** 2026-05-01, against `main` at commit
`450c0b1` (post-rename: `μ-v8` → `OPD`, `σ` → `baseline`,
`grpo_v1` → `grpo`).


## §2  Shared building blocks

Three things are shared across (some or all of) the trainers. Pulling
them out here means each per-trainer flowchart can stay focused on what
is distinctive.

### 2.1  Tinker client topology

| Client | Where it lives | Used by | Frozen? |
|---|---|---|---|
| `service_client` | `train_*.py` (created via `create_service_client`) | all | n/a — auth + factory |
| `training_client` | LoRA r=64, Adam(lr=5e-5, β=0.9/0.95) | GRPO, OPD | trainable |
| `sampling_client` | created from `training_client.save_weights_for_sampler` each iter | GRPO, OPD | reflects current LoRA |
| `sampling_client` (frozen variant) | `service_client.create_sampling_client(base_model=...)` | baseline | frozen base weights |
| `grader_client` | frozen `Qwen/Qwen3-30B-A3B`, T=0.0 | baseline, GRPO | frozen |

Code: `train_opd.py:296-323` (training client init), `train_opd.py:325-327`
(Adam params), `train_grpo.py:212-244` (same pattern), `train_baseline.py:122-126`
(frozen sampling + grader).

### 2.2  File-bus subagent pattern (OPD only)

OPD does not call any HTTP-style API for the Opus reviewer. It uses an
on-disk message bus that a separate Claude Code window polls. The
trainer is the producer; the subagent in the other window is the
consumer.

```
trainer (this process)                         subagent (other window)
    │                                                │
    │ critic_client.submit(iter_idx, payload)        │
    │   atomic write of                              │
    │   runs/{run}/critic_requests/iter_NNN.json     │
    │ ───────────────────────────────────────────►   │
    │                                                │ poll dir every 30s
    │                                                │ read req → invoke Opus 4.7
    │                                                │   with reference_proposal.md
    │                                                │   as PRIVILEGED context
    │                                                │ write
    │                                                │   runs/{run}/critic_responses/iter_NNN.json
    │ critic_client.wait(iter_idx)                   │
    │   blocks polling resp file                     │
    │   up to 900s; falls back to                    │
    │   COLD_START_CRITIQUE_NONE on timeout          │
    │   (train_opd.py:411-418)                       │
```

Implementation: `src/co_scientist/shared/opus_critic_subagent.py`
(`OpusCriticClient.submit/wait`); subagent runbook:
`projects/d6_grant_proposal/scripts/reviewer_subagent.md`.

The audit subagent (`OpusAuditClient`) follows the same pattern but is
**fire-and-forget** — `submit` is called per iter, results are
collected at the end of training (`train_opd.py:663-680`).

### 2.3  Prompt registry (single source of truth)

D6 externalises every production prompt to
`projects/d6_grant_proposal/prompts/`. The loader
(`src/co_scientist/d6_grant_proposal/prompt_loader.py`) reads the file,
strips the doc header above the first `---` rule, and applies
`str.format(**kwargs)`. Editing a `.md` propagates to the next training
run with no code change.

| Path under `prompts/` | Role | Template variables | Loaded by |
|---|---|---|---|
| `generation/student.md` | Student rollout / sampling context (no critique) | `{goal}`, `{oracle_abstraction}`, `{external_knowledge}` | baseline, GRPO, OPD |
| `generation/teacher.md` | Teacher logprob context (with critique) | `{goal}`, `{oracle_abstraction}`, `{external_knowledge}`, `{critique_xml}` | OPD only |
| `generation/cold_start_critique.md` | Iter-0 critique placeholder | (none) | OPD only |
| `review/critic_instructions.md` | Opus critic system instructions (8-child schema) | (none) | OPD only |
| `audit/absolute_score.md` | 12-signal Opus audit prompt | (per-payload) | audit *daemon*, not the trainers' inline path |
| `pairwise/head_to_head.md` | Pairwise A/B verdict over 12 dims | (per-payload) | none currently — caller deferred |
| `oracle/extract_items.md` | Oracle item extraction | (per-paper) | `extract_oracle.py` (one-time, not at train time) |

Builder helpers in `src/co_scientist/d6_grant_proposal/prompts.py:36-97`:
`build_student_prompt`, `build_teacher_prompt`, `build_critic_request_payload`,
`build_audit_request_payload`. The `COLD_START_CRITIQUE_NONE` constant is
loaded once at module import (`prompts.py:70`).

### 2.4  The 12-signal grader prompt is **not** a `.md` file

The Qwen3-30B-A3B grader prompt used by baseline and GRPO is built in
**code** by `build_single_call_prompt(goal, plan_text[:15000], active_signals)`
inside `src/co_scientist/shared/grant_signal_reward.py`. Looking under
`prompts/` for it will fail. The function inlines all 12 signal
specifications (questions, COT scaffolding, 1–5 rubrics) into one XML
template. Active-signal selection (G12 vs G12a for non-STEM) lives in
`_active_signals(goal_domain)` inside each trainer
(`train_baseline.py:54-62`, `train_grpo.py:88-96`).

OPD does **not** call this prompt during training — its only quality
signal is the Opus reviewer's `<critique>` and the (async, fire-and-forget)
12-signal audit. The Qwen 12-signal grader appears only in baseline +
GRPO + OPD's eval audit daemon.


## §3  `train_baseline` — frozen lower bound

**Role.** Establishes the score the OPD/GRPO trainers must beat. No
gradient step, no LoRA — frozen Qwen3-30B-A3B samples
`n_eval_plans=8` plans against the slim oracle, frozen Qwen3-30B-A3B
grades each, results land in `buffer.jsonl` + `metrics.jsonl`. Decision
threshold: OPD/GRPO must exceed baseline+0.02 weighted-mean to claim
improvement (`train_baseline.py:219`).

### 3.1  Pre-loop setup

<div align="center">

<img src="figures/baseline_setup.svg" alt="Baseline pre-loop setup" />

</div>

<details><summary>Mermaid source (baseline_setup.mmd)</summary>

```mermaid
flowchart TD
    GoalMD[research_goal.md]:::io
    OracleMD[oracle/slim.md]:::io
    Cfg[Config<br/>n_eval_plans=8<br/>train_baseline.py:65-89]:::cfg
    Renderer[get_tokenizer + renderers.get_renderer<br/>:118-120]:::infra
    SC[create_service_client<br/>:122-124]:::infra
    SampClient[sampling_client<br/>FROZEN Qwen3-30B-A3B<br/>:125]:::frozenllm
    GradClient[grader_client<br/>FROZEN Qwen3-30B-A3B, T=0.0<br/>:126]:::frozenllm
    AS[_active_signals goal_domain<br/>:128]:::infra
    BSP[build_student_prompt<br/>prompts/generation/student.md<br/>:130]:::prompt
    SI[renderer.build_generation_prompt<br/>:132]:::infra
    SP[SamplingParams<br/>max=8192, T=1.0, top_p=0.95<br/>:134-139]:::cfg
    GP[grader SamplingParams<br/>max=8192, T=0.0<br/>:140-144]:::cfg

    GoalMD -->|goal:str| BSP
    OracleMD -->|oracle:str| BSP
    BSP -->|student_text| SI
    SI -->|student_input ModelInput| RolloutPhase((to §3.2))
    SC --> SampClient
    SC --> GradClient
    Cfg --> Renderer
    Renderer --> SI

    classDef io fill:#f7f7f7,stroke:#666,color:#222
    classDef cfg fill:#f0f0fa,stroke:#5b3aaa,color:#2a1a55
    classDef prompt fill:#eef7ff,stroke:#1a5fb4,color:#0d3b66
    classDef infra fill:#f5f5f5,stroke:#888,color:#222
    classDef frozenllm fill:#fff5e6,stroke:#cc7700,color:#664400
```

</details>

### 3.2  Per-pass flow (single non-iterative pass)

<div align="center">

<img src="figures/baseline_loop.svg" alt="Baseline per-pass flow" />

</div>

<details><summary>Mermaid source (baseline_loop.mmd)</summary>

```mermaid
flowchart TD
    Start([single pass<br/>k=0..n_eval_plans-1])
    Sample[sampling_client.sample<br/>num_samples=8<br/>FROZEN Qwen3-30B-A3B<br/>train_baseline.py:147-151]:::llm
    Decode[parse_response + extract_solution<br/>per plan k<br/>:158-162]:::infra
    BuildG[build_single_call_prompt<br/>shared/grant_signal_reward.py<br/>NOT a .md file<br/>:165]:::prompt
    Grade[grader_client.sample<br/>FROZEN Qwen3-30B-A3B, T=0.0<br/>:169-171]:::llm
    Parse[parse_scores + aggregate_reward<br/>D4 12-signal weighted mean<br/>:175-177]:::infra
    Record[append to records list<br/>:184-194]:::infra
    Buf[buffer.jsonl write<br/>:197-200]:::io
    Met[metrics.jsonl write<br/>:202-215]:::io
    NoGrad{{NO gradient step<br/>NO LoRA update<br/>:1-10 docstring}}:::warn

    Start --> Sample
    Sample -->|seq.tokens| Decode
    Decode -->|plan_text| BuildG
    BuildG -->|grader_prompt_str + active_signals 12| Grade
    Grade -->|grader_text XML| Parse
    Parse -->|reward 0-1| Record
    Record -->|after k=7| Buf
    Buf --> Met
    Sample -.-> NoGrad

    classDef llm fill:#fff5e6,stroke:#cc7700,color:#664400
    classDef io fill:#f7f7f7,stroke:#666,color:#222
    classDef infra fill:#f5f5f5,stroke:#888,color:#222
    classDef prompt fill:#eef7ff,stroke:#1a5fb4,color:#0d3b66
    classDef warn fill:#fff,stroke:#c0392b,stroke-dasharray:5 3,color:#c0392b,stroke-width:2px
```

</details>

### 3.3  Prompt inventory

| Path | Role | Loaded at | Vars |
|---|---|---|---|
| `prompts/generation/student.md` | Student context (no critique) | `train_baseline.py:130` via `build_student_prompt(goal, oracle)` | `{goal}`, `{oracle_abstraction}`, `{external_knowledge}` (defaults to `(none provided for this run)`) |
| `build_single_call_prompt` (CODE, not `.md`) | Qwen3 12-signal grader prompt | `train_baseline.py:165` | dynamic over `active_signals` |

Templates **NOT** loaded by this trainer: teacher.md, cold_start_critique.md,
critic_instructions.md, audit/absolute_score.md, pairwise/head_to_head.md.

### 3.4  Data flow

| Read-only inputs | Per-pass writes | State across iters |
|---|---|---|
| `dataset/{domain}/{goal}/research_goal.md` | `runs/{run}/buffer.jsonl` (8 lines, one per plan) | none — single pass |
| `data/{domain}/{goal}/oracle/slim.md` | `runs/{run}/metrics.jsonl` (1 summary line) |  |
|  | `runs/{run}/config.json` |  |

### 3.5  No gradient step (verbatim)

```python
# train_baseline.py:1-10 (docstring)
"""D6 frozen baseline — frozen Qwen3-30B-A3B + oracle, no training.

Generates n_eval_plans proposals using the frozen base model with the student
prompt (goal + oracle abstraction). Scores each plan with the 12-signal grant
reward and writes buffer.jsonl + metrics.jsonl in the same schema as train_opd.py,
so the same audit.py analysis tools apply.

This is the baseline arm of the paper: what you get from the base model + oracle
without any gradient updates. ...
"""
```

No `import torch` for tensors of trainable params, no `optim_step`, no
`forward_backward`, no `training_client`. Both `sampling_client` and
`grader_client` are inference-only clients
(`train_baseline.py:122-126`).

### 3.6  Doc-vs-code consistency (`pipeline_docs/train_baseline.md`)

- ✅ Purpose, no-training claim, 12-signal grader, output paths — all match.
- ⚠️ **Drift**: line 43 still says *"Sigma's job"* — residual from the
  `σ → baseline` rename (DECISIONS.md 2026-04-30 §"D6 naming
  standardization"). Cosmetic; flag for a doc commit.


## §4  `train_grpo` — reward-only ablation

**Role.** Same training-loop machinery as OPD but **no Opus reviewer
and no teacher**. Reward signal is the Qwen3-30B 12-signal aggregate;
advantage is group-relative and broadcast (one scalar per plan,
copied to every generated token). The OPD-vs-GRPO comparison is the
paper's load-bearing claim about whether the privileged reviewer
matters.

`train_grpo.py:200` logs `[NO Opus reviewer — reward-only GRPO]` —
this is the textual flag in run logs that the GRPO arm is active.

### 4.1  Pre-loop setup

<div align="center">

<img src="figures/grpo_setup.svg" alt="GRPO pre-loop setup" />

</div>

<details><summary>Mermaid source (grpo_setup.mmd)</summary>

```mermaid
flowchart TD
    GoalMD[research_goal.md]:::io
    OracleMD[oracle/slim.md]:::io
    Cfg[Config<br/>n_rollouts_per_group=4<br/>lora_rank=64<br/>train_grpo.py:99-146]:::cfg
    Renderer[tokenizer + renderer<br/>:208-210]:::infra
    SC[create_service_client<br/>:212-214]:::infra
    GradClient[grader_client<br/>FROZEN Qwen3-30B-A3B<br/>:217]:::frozenllm
    Train[training_client<br/>LoRA r=64, fresh OR resume<br/>:220-240]:::trainllm
    Adam[AdamParams<br/>lr=5e-5, β=0.9/0.95, ε=1e-8<br/>:242-244]:::cfg
    AS[_active_signals goal_domain<br/>:257]:::infra
    BSP[build_student_prompt<br/>prompts/generation/student.md<br/>:259]:::prompt
    SI[renderer.build_generation_prompt<br/>+ to_ints<br/>:261-262]:::infra
    SP[SamplingParams<br/>max=8192, T=1.0, top_p=0.95<br/>:245-250]:::cfg
    GP[grader SamplingParams<br/>:251-255]:::cfg
    Paths[buffer/metrics/checkpoints/eval paths<br/>:264-267]:::io

    GoalMD -->|goal:str| BSP
    OracleMD -->|oracle:str| BSP
    BSP -->|student_text| SI
    SI -->|student_input + student_tokens_prefix| LoopStart((to §4.2))
    SC --> Train
    SC --> GradClient
    Cfg --> Renderer
    Cfg --> Adam

    classDef io fill:#f7f7f7,stroke:#666,color:#222
    classDef cfg fill:#f0f0fa,stroke:#5b3aaa,color:#2a1a55
    classDef prompt fill:#eef7ff,stroke:#1a5fb4,color:#0d3b66
    classDef infra fill:#f5f5f5,stroke:#888,color:#222
    classDef frozenllm fill:#fff5e6,stroke:#cc7700,color:#664400
    classDef trainllm fill:#fff0f0,stroke:#cc0044,color:#660022
```

</details>

### 4.2  Per-iteration flow

<div align="center">

<img src="figures/grpo_loop.svg" alt="GRPO per-iteration flow" />

</div>

<details><summary>Mermaid source (grpo_loop.mmd)</summary>

```mermaid
flowchart TD
    Iter([Iter idx])
    Ckpt[Save checkpoint<br/>if iter %% save_every == 0<br/>:275-283]:::infra
    Wts[save_weights_for_sampler<br/>create new sampling_client<br/>:285-291]:::infra
    Sample[sampling_client.sample<br/>n_rollouts_per_group=4<br/>via prompts/generation/student.md<br/>:295-302]:::llm
    Decode[parse_response + extract_solution × 4<br/>:305-312]:::infra
    Grade[_grade_plan × 4<br/>build_single_call_prompt<br/>FROZEN Qwen3-30B grader<br/>:314-326]:::llm
    AdvCalc["group advantage<br/>(r_i − mean) / (std + 1e-8)<br/>:333-335"]:::infra
    Skip{rewards all 0?<br/>OR all advs 0?<br/>:329-339}:::infra
    Clip["clip ±5<br/>broadcast to all gen tokens<br/>per_tok_adv = [a]*len(gen)<br/>:360-361"]:::infra
    Mask[solution_only_mask<br/>build_solution_token_mask_from_tokens<br/>:366-373]:::infra
    Datum[build types.Datum × 4<br/>target_tokens, logprobs, advantages<br/>:385-399]:::infra
    Loss["forward_backward + optim_step<br/>×4 grad steps over chunks<br/>loss_fn=ppo, ε=0.2<br/>:415-429"]:::gradient
    Eval[Eval rollout n=8<br/>append eval_rollouts.jsonl<br/>:433-450]:::llm
    Buf[buffer.jsonl + metrics.jsonl<br/>:452-475]:::io

    Iter --> Ckpt --> Wts --> Sample --> Decode --> Grade --> AdvCalc --> Skip
    Skip -- continue --> Clip
    Skip -- "skip iter" --> Iter
    Clip --> Mask --> Datum --> Loss --> Eval --> Buf

    classDef llm fill:#fff5e6,stroke:#cc7700,color:#664400
    classDef gradient fill:#fff5f5,stroke:#c0392b,stroke-width:2.5px,color:#7a0000
    classDef io fill:#f7f7f7,stroke:#666,color:#222
    classDef infra fill:#f5f5f5,stroke:#888,color:#222
```

</details>

### 4.3  Prompt inventory

| Path | Role | Loaded at | Vars |
|---|---|---|---|
| `prompts/generation/student.md` | Student rollout context | `train_grpo.py:259` via `build_student_prompt` | `{goal}`, `{oracle_abstraction}`, `{external_knowledge}` |
| `build_single_call_prompt` (CODE) | Qwen3 12-signal grader prompt | `_grade_plan` at `train_grpo.py:159` | dynamic over `active_signals` |

**Not loaded**: teacher.md, cold_start_critique.md, critic_instructions.md,
audit/absolute_score.md, pairwise/head_to_head.md. The trainer's
`generation/teacher.md` and `cold_start_critique.md` are not imported —
no `build_teacher_prompt` call, no `OpusCriticClient`.

### 4.4  Data flow

| Read-only inputs | Per-iter writes | State across iters |
|---|---|---|
| `dataset/{domain}/{goal}/research_goal.md` | `runs/{run}/buffer.jsonl` (4 lines/iter) | LoRA weights in `training_client` |
| `data/{domain}/{goal}/oracle/slim.md` | `runs/{run}/metrics.jsonl` (1 line/iter) | Adam optimizer state |
|  | `runs/{run}/eval_rollouts.jsonl` (8 lines/iter) | (resumable via `last_checkpoint`) |
|  | `runs/{run}/checkpoints.jsonl` |  |
|  | `runs/{run}/config.json` (once) |  |

NOTE: no `critic_requests/`, no `critic_responses/`, no
`audit_requests/` — confirmed by absence of `OpusCriticClient` /
`OpusAuditClient` imports in `train_grpo.py:53-65`.

### 4.5  Loss / advantage (verbatim)

```python
# train_grpo.py:333-335 — group-relative advantage
mean_r = float(np.mean(rewards))
std_r = float(np.std(rewards)) + config.adv_norm_eps      # adv_norm_eps=1e-8
advantages_scalar = [(r - mean_r) / std_r for r in rewards]

# train_grpo.py:360-361 — clip + broadcast to per-token
clipped_adv = max(-config.reward_clip, min(config.reward_clip, adv_scalar))
per_tok_adv = [clipped_adv] * len(gen)

# train_grpo.py:366-373 — solution-only mask (same as OPD)
if config.solution_only_mask:                              # default True
    sol_mask = build_solution_token_mask_from_tokens(
        tokenizer=tokenizer, sample_tokens=gen,
    )
    if sol_mask.size == len(per_tok_adv):
        per_tok_adv = [a * float(m) for a, m in zip(per_tok_adv, sol_mask.tolist())]

# train_grpo.py:419-429 — PPO-clip optimizer step
loss_fn_kwargs = {"loss_fn": config.loss_fn_name}          # "ppo"
if config.loss_fn_name == "ppo":
    loss_fn_kwargs["loss_fn_config"] = {
        "clip_low_threshold": 1.0 - config.ppo_clip_eps,   # 0.8
        "clip_high_threshold": 1.0 + config.ppo_clip_eps,  # 1.2
    }
for chunk in chunks[:n_steps]:                             # n_steps = 4
    if not chunk:
        continue
    training_client.forward_backward(chunk, **loss_fn_kwargs).result()
    training_client.optim_step(adam_params).result()
```

### 4.6  Doc-vs-code consistency (`pipeline_docs/train_grpo.md`)

- ✅ "no Opus reviewer and no SDPO" claim — confirmed by `:200` log line.
- ✅ Advantage formula in doc matches `:333-335, :360`.
- ✅ `solution_only_mask=True`, `ppo_clip_eps=0.2`, `loss_fn_name="ppo"`
  defaults match `Config` at `:137-139`.
- ⚠️ **Drift**: doc line 50 lists `weighted_aggregate` as a dependency
  function; the actual function imported at `:54-56` is
  `aggregate_reward`. Cosmetic — same behaviour, stale name.


## §5  `train_opd` — paper main arm (privileged-Opus OPD)

**Role.** D5's option-(c) HER SDPO recipe applied to grant proposals.
Three load-bearing components:

1. **Student context** = `goal + slim_oracle` (no critique).
2. **Teacher context** = `goal + slim_oracle + critique` — only the
   `<improvement_directive>` child is the active conditioning signal,
   the other 7 are debug context.
3. **Privileged-Opus reviewer** sees the rollout *plus*
   `reference_proposal.md` as ground truth (the student never sees it),
   emits a structured `<critique>` via the file-bus subagent.

Per-token advantage = `clamp(sdpo_scale*(teacher_lp − student_lp), ±5)`.
Solution-only mask zeros the advantage outside the last
`<solution>...</solution>` body. PPO-clip loss with ε=0.2.

**Trust-region α-blend was REMOVED 2026-04-30** —
`train_opd.py:16-20` documents the removal. No frozen-base forward
pass, no α weighted average. The shipped code IS the recipe.

### 5.1  Pre-loop setup

<div align="center">

<img src="figures/opd_setup.svg" alt="OPD pre-loop setup" />

</div>

<details><summary>Mermaid source (opd_setup.mmd)</summary>

```mermaid
flowchart TD
    GoalMD[research_goal.md]:::io
    RefMD["reference_proposal.md<br/>PRIVILEGED — student never sees this<br/>train_opd.py:257"]:::priv
    OracleMD[oracle/slim.md]:::io
    Cfg[Config<br/>n_plans=8 lora_rank=64<br/>opd_anchor_clip=5.0<br/>ppo_clip_eps=0.2<br/>:85-136]:::cfg
    Cold[COLD_START_CRITIQUE_NONE<br/>prompts/generation/cold_start_critique.md<br/>prompts.py:70]:::prompt
    Renderer[tokenizer + renderer<br/>:291-294]:::infra
    SC[create_service_client<br/>:296-298]:::infra
    Train[training_client<br/>LoRA r=64<br/>fresh / resume / external init<br/>:301-323]:::trainllm
    Adam[AdamParams<br/>lr=5e-5, β=0.9/0.95<br/>:325-327]:::cfg
    SP[SamplingParams<br/>max=8192, T=1.0, top_p=0.95<br/>:328-333]:::cfg
    Critic["OpusCriticClient<br/>file-bus, timeout=900s<br/>writes critic_requests/iter_NNN.json<br/>:335"]:::opus
    Audit["OpusAuditClient<br/>file-bus, fire-and-forget<br/>:336"]:::opus
    Paths[buffer/metrics/checkpoints paths<br/>:338-340]:::io

    GoalMD -->|goal| Loop((to §5.2))
    OracleMD -->|oracle| Loop
    RefMD -->|reference_proposal_md| Loop
    Cold -->|critique_current iter 0| Loop
    SC --> Train
    SC --> Critic
    SC --> Audit

    classDef io fill:#f7f7f7,stroke:#666,color:#222
    classDef priv fill:#ffe6e6,stroke:#aa0000,stroke-width:2px,color:#660000
    classDef cfg fill:#f0f0fa,stroke:#5b3aaa,color:#2a1a55
    classDef prompt fill:#eef7ff,stroke:#1a5fb4,color:#0d3b66
    classDef infra fill:#f5f5f5,stroke:#888,color:#222
    classDef trainllm fill:#fff0f0,stroke:#cc0044,color:#660022
    classDef opus fill:#e6e6ff,stroke:#5500aa,stroke-width:2px,color:#220066
```

</details>

### 5.2  Per-iteration flow

<div align="center">

<img src="figures/opd_loop.svg" alt="OPD per-iteration flow" />

</div>

<details><summary>Mermaid source (opd_loop.mmd)</summary>

```mermaid
flowchart TD
    Iter([Iter idx])
    Ckpt[Save checkpoint?<br/>train_opd.py:349-357]:::infra
    Wts[save_weights_for_sampler<br/>:360-366]:::infra
    StuP[build_student_prompt<br/>prompts/generation/student.md<br/>:369-372]:::prompt
    Sample[sampling_client.sample<br/>n_plans=8<br/>:374-380]:::llm
    Dec[_decode_plan × n_plans<br/>:386-392]:::infra
    Pick[rng.choice valid_idxs<br/>one plan to critique<br/>:399-401]:::infra
    Pay[build_critic_request_payload<br/>+ reference_proposal_md PRIVILEGED<br/>+ prompts/review/critic_instructions.md<br/>:402-408]:::prompt
    Sub[critic_client.submit<br/>writes critic_requests/iter_NNN.json<br/>:410]:::opus
    Wait["critic_client.wait BLOCKING<br/>polls critic_responses/iter_NNN.json<br/>up to 900s<br/>OPUS 4.7 subagent<br/>:411-415"]:::opus
    Fb{timeout?}:::infra
    Cold2[COLD_START_CRITIQUE_NONE<br/>:416-418]:::prompt
    TchP["build_teacher_prompt<br/>prompts/generation/teacher.md<br/>+ critique_xml<br/>:426-429"]:::prompt
    TLP["compute_logprobs<br/>teacher prefix + student tokens<br/>:432-441"]:::llm
    SLP[student logprobs<br/>already in seq.logprobs<br/>:449]:::infra
    Adv["per-token advantage<br/>clamp(sdpo_scale*(t_lp − s_lp), ±5)<br/>:463-468"]:::infra
    Mask["solution_only_mask<br/>build_solution_token_mask_from_tokens<br/>zeros advantage outside &lt;solution&gt;<br/>:471-485"]:::infra
    Dat[build_sdpo_datum × n<br/>:488-494]:::infra
    PPO["forward_backward + optim_step<br/>×4 grad steps over chunks<br/>loss_fn=ppo ε=0.2<br/>:508-531"]:::gradient
    Buf[buffer.jsonl<br/>:542-544]:::io
    EvalSamp[eval rollout n=8<br/>:548-554]:::llm
    AudPay[build_audit_request_payload<br/>NO reference here<br/>prompt_version=d6_v1<br/>:560-565]:::prompt
    AudSub[audit_client.submit ASYNC<br/>fire-and-forget<br/>:566]:::opus
    EvalRoll[eval_rollouts.jsonl<br/>:568-571]:::io
    Met[metrics.jsonl<br/>:579-602]:::io
    ES[early-stop guard<br/>read audit_responses/<br/>halt if Δ ≥ 5.0<br/>:607-647]:::infra

    Iter --> Ckpt --> Wts --> StuP --> Sample
    Sample -->|seq.tokens × n| Dec --> Pick --> Pay --> Sub --> Wait --> Fb
    Fb -- "ok" --> TchP
    Fb -- "timeout" --> Cold2 --> TchP
    TchP --> TLP
    Sample --> SLP
    TLP -->|t_lp_seq per token| Adv
    SLP -->|s_lp per token| Adv
    Adv --> Mask --> Dat --> PPO --> Buf --> EvalSamp --> AudPay --> AudSub
    AudSub -.->|async| Met
    EvalSamp --> EvalRoll
    Buf --> Met --> ES

    classDef llm fill:#fff5e6,stroke:#cc7700,color:#664400
    classDef opus fill:#e6e6ff,stroke:#5500aa,stroke-width:2px,color:#220066
    classDef gradient fill:#fff5f5,stroke:#c0392b,stroke-width:2.5px,color:#7a0000
    classDef prompt fill:#eef7ff,stroke:#1a5fb4,color:#0d3b66
    classDef io fill:#f7f7f7,stroke:#666,color:#222
    classDef infra fill:#f5f5f5,stroke:#888,color:#222
```

</details>

### 5.3  Prompt inventory

| Path | Role | Loaded at | Vars |
|---|---|---|---|
| `prompts/generation/student.md` | Student rollout context | `train_opd.py:369` via `build_student_prompt(goal, oracle)` | `{goal}`, `{oracle_abstraction}`, `{external_knowledge}` |
| `prompts/generation/teacher.md` | Teacher logprob context (with critique) | `train_opd.py:426` via `build_teacher_prompt(goal, oracle, critique_current)` | `{goal}`, `{oracle_abstraction}`, `{external_knowledge}`, `{critique_xml}` |
| `prompts/generation/cold_start_critique.md` | Iter-0 critique placeholder + timeout fallback | imported at `prompts.py:70` as `COLD_START_CRITIQUE_NONE`; used at `train_opd.py:397, 414, 418` | (none) |
| `prompts/review/critic_instructions.md` | Opus critic system instructions (8-child schema) | inlined into payload at `prompts.py:96` (called from `train_opd.py:402-408`) | (none) |
| `build_audit_request_payload` (no `.md` text — payload structure only) | Audit request shape | `train_opd.py:560-565` | iter, goal, plans, goal_id |

The `audit/absolute_score.md` template is **not loaded by this trainer**;
it lives in the audit subagent's runbook (the trainer just submits a
structural payload).

### 5.4  Data flow

| Read-only inputs (per run) | Per-iter writes | State across iters |
|---|---|---|
| `dataset/{domain}/{goal}/research_goal.md` | `runs/{run}/buffer.jsonl` (n_plans lines/iter) | LoRA weights in `training_client` |
| `dataset/{domain}/{goal}/reference_proposal.md` (privileged → critic only) | `runs/{run}/metrics.jsonl` (1 line/iter) | Adam optimizer state |
| `data/{domain}/{goal}/oracle/slim.md` | `runs/{run}/checkpoints.jsonl` | (resumable via `last_checkpoint`) |
| `prompts/generation/cold_start_critique.md` (iter 0 fallback) | `runs/{run}/critic_requests/iter_NNN.json` |  |
|  | `runs/{run}/critic_responses/iter_NNN.json` (written by subagent) |  |
|  | `runs/{run}/audit_requests/iter_NNN.json` |  |
|  | `runs/{run}/audit_responses/iter_NNN.json` (async, written by subagent) |  |
|  | `runs/{run}/eval_rollouts.jsonl` |  |
|  | `runs/{run}/audit_log.jsonl` (final, post-loop collect) |  |

### 5.5  Loss / advantage (verbatim)

```python
# train_opd.py:463-468 — privileged-Opus per-token advantage
clip_v = config.opd_anchor_clip                            # 5.0
per_tok_adv = []
for t_, s_ in zip(t_lp_seq, s_lp):
    a = float(t_ - s_) * config.sdpo_scale                 # sdpo_scale=1.0
    a = max(-clip_v, min(clip_v, a))
    per_tok_adv.append(a)

# train_opd.py:471-485 — solution-only mask
n_mask_active = len(per_tok_adv)
mask_density = 1.0
if config.solution_only_mask:                              # default True
    sol_mask = build_solution_token_mask_from_tokens(
        tokenizer=tokenizer, sample_tokens=gen,
    )
    if sol_mask.size == len(per_tok_adv):
        per_tok_adv = [a * float(m) for a, m in zip(per_tok_adv, sol_mask.tolist())]
        n_mask_active = int(sol_mask.sum())
        mask_density = float(n_mask_active) / max(1, len(gen))

# train_opd.py:514-528 — PPO-clip optimizer step
loss_fn_kwargs = {"loss_fn": config.loss_fn_name}          # "ppo"
if config.loss_fn_name == "ppo":
    loss_fn_kwargs["loss_fn_config"] = {
        "clip_low_threshold": 1.0 - config.ppo_clip_eps,   # 0.8
        "clip_high_threshold": 1.0 + config.ppo_clip_eps,  # 1.2
    }
for chunk in chunks[:n_steps]:                             # n_steps = 4
    if not chunk:
        continue
    fb_fut = training_client.forward_backward(chunk, **loss_fn_kwargs)
    os_fut = training_client.optim_step(adam_params)
    fb_fut.result()
    os_fut.result()
```

The mask-construction internals are at `train_opd.py:177-234` —
`find_solution_content_span` finds the **last** non-think-overlapping
`<solution>...</solution>` body via regex, then maps the char span
to a token span via re-encoding.

### 5.6  Reviewer file-bus mechanics

Mapping `train_opd.py` ↔ `scripts/reviewer_subagent.md`:

| Step | Trainer (this process) | Subagent (other window) |
|---|---|---|
| produce request | `:410` `critic_client.submit(iter_idx, payload)` writes `critic_requests/iter_NNN.json` | poll dir every 30s (runbook §"Loop protocol") |
| privileged context | `payload["source_paper_md"] = reference_proposal_md` (`prompts.py:94`) | runbook line 16-20: "DO NOT reveal the reference proposal verbatim or by name" |
| produce response | `:411-415` `critic_client.wait` polls `critic_responses/iter_NNN.json`, blocks up to 900s | runbook §"Output format" — JSON with `judgments: {plan_id: {critique_xml}}` |
| timeout | `:416-418` falls back to `COLD_START_CRITIQUE_NONE` and continues | runbook §"Loop protocol": stop after 20 min idle |

### 5.7  Doc-vs-code consistency (`pipeline_docs/train_opd.md`)

- ✅ Three components (sol mask / privileged-Opus offset / PPO-clip),
  trust-region α removal, advantage formula, privileged-reviewer story
  — all match.
- ⚠️ **Drift 1**: doc line 65 references `configs/pilot_foundrl_v8.json`
  — actual filename after the 2026-04-30 rename is
  `configs/pilot_foundrl_opd.json`. Cosmetic.
- ⚠️ **Drift 2**: `scripts/reviewer_subagent.md:51-58` documents the
  **5-child** critique schema (`idea_alignment`,
  `missing_components`, `incorrect_assumptions`, `feasibility`,
  `improvement_directive`); the production prompt
  `prompts/review/critic_instructions.md:25-34` is the **8-child**
  schema (adds `clarity`, `strengths`, `weaknesses` per the
  2026-04-30 prompt-content commit). The trainer follows
  `critic_instructions.md`, so the runbook is the stale side. The
  added children do not change the OPD distillation gradient — only
  `<improvement_directive>` is read by the teacher prompt
  (`prompts/generation/teacher.md:31` injects `{critique_xml}`,
  but the teacher reads only `improvement_directive` per
  `critic_instructions.md:13-15`). Operationally harmless; doc fix
  needed for paper.


## §6  Cross-arm side-by-side

The three arms aligned by stage. Components that exist **only** in OPD
are visually distinct (purple = privileged-Opus, salmon = teacher–student
delta machinery, dashed = async). The "is the reviewer load-bearing?"
question reduces to whether the OPD-only column lifts performance over
the GRPO column at fixed loss machinery (sol mask + PPO-clip + Adam are
identical between GRPO and OPD).

<div align="center">

<img src="figures/cross_arm.svg" alt="Cross-arm side-by-side comparison" />

</div>

<details><summary>Mermaid source (cross_arm.mmd)</summary>

```mermaid
flowchart LR
    subgraph BL[baseline ─ frozen]
        direction TB
        b1[load goal+oracle] --> b2[student.md]:::prompt
        b2 --> b3[sample n=8<br/>FROZEN Qwen3]:::frozenllm
        b3 --> b4[Qwen3 12-signal grader]:::frozenllm
        b4 --> b5[buffer.jsonl]:::io
    end

    subgraph GR[GRPO ─ reward-only]
        direction TB
        g1[load goal+oracle] --> g2[student.md]:::prompt
        g2 --> g3[sample n=4<br/>LoRA Qwen3]:::trainllm
        g3 --> g4[Qwen3 12-signal grader]:::frozenllm
        g4 --> g5["group adv<br/>(r−μ)/σ"]:::infra
        g5 --> g6[sol mask]:::infra
        g6 --> g7[PPO-clip step ε=0.2]:::gradient
        g7 --> g8[buffer.jsonl]:::io
    end

    subgraph OP[OPD ─ paper main arm]
        direction TB
        o0["reference_proposal.md<br/>PRIVILEGED"]:::priv
        o1[load goal+oracle] --> o2[student.md]:::prompt
        o2 --> o3[sample n=8<br/>LoRA Qwen3]:::trainllm
        o3 --> o4[OPUS 4.7 critic<br/>file-bus subagent<br/>+ critic_instructions.md]:::opus
        o0 --> o4
        o4 --> o5[teacher.md<br/>+ critique_xml]:::opd
        o5 --> o6[teacher logprobs]:::opd
        o3 --> o7[student logprobs]:::opd
        o6 --> o8["per-tok adv<br/>clamp(t_lp−s_lp, ±5)"]:::opd
        o7 --> o8
        o8 --> o9[sol mask]:::infra
        o9 --> o10[PPO-clip step ε=0.2]:::gradient
        o10 --> o11[buffer.jsonl]:::io
        o10 -.->|async| o12[Opus audit]:::async
    end

    classDef io fill:#f7f7f7,stroke:#666,color:#222
    classDef prompt fill:#eef7ff,stroke:#1a5fb4,color:#0d3b66
    classDef frozenllm fill:#fff5e6,stroke:#cc7700,color:#664400
    classDef trainllm fill:#fff0f0,stroke:#cc0044,color:#660022
    classDef gradient fill:#fff5f5,stroke:#c0392b,stroke-width:2.5px,color:#7a0000
    classDef opus fill:#e6e6ff,stroke:#5500aa,stroke-width:2px,color:#220066
    classDef opd fill:#fff7e6,stroke:#cc7700,color:#664400
    classDef priv fill:#ffe6e6,stroke:#aa0000,stroke-width:2px,color:#660000
    classDef async fill:#fffbe6,stroke:#aa8800,stroke-dasharray:5 3,color:#5e4500
    classDef infra fill:#f5f5f5,stroke:#888,color:#222
```

</details>

**OPD-only nodes** (absent from baseline and GRPO lanes):

- `reference_proposal.md` privileged input
- Opus 4.7 critic call (file-bus subagent)
- `teacher.md` prompt (with `{critique_xml}` slot)
- Teacher logprob compute over student tokens under teacher context
- Per-token advantage `clamp(teacher_lp − student_lp, ±5)`
- Async Opus audit

**Identical between GRPO and OPD** (not the differentiator):

- LoRA r=64, lr=5e-5, Adam(β=0.9/0.95)
- Solution-only token mask (`build_solution_token_mask_from_tokens`)
- PPO-clip loss with ε=0.2
- 4 grad steps per iter
- buffer/metrics/checkpoints persistence schema


## §7  How to use this notebook

- **Reading code.** Every box / table cell in this notebook carries a
  `path:line` anchor. Click into the editor at that line to see the
  exact code behind the abstraction.
- **Editing a prompt.** Locate the template in §2.3 → the "Loaded by"
  column tells you which trainers consume it → audit each trainer's
  per-iter flowchart (§3.2 / §4.2 / §5.2) for how the loaded text
  enters the gradient path. Then `reload_prompts()` for any live dev
  loop (`prompt_loader.py:75`).
- **Adding a new arm.** Copy the §3 / §4 / §5 sub-template:
  setup flowchart → per-iter flowchart → prompt inventory → data flow
  → loss block → consistency footer. Cross-arm cell §6 should grow a
  fourth lane.
- **Verifying pilot config.** `configs/pilot_foundrl_{baseline,grpo,opd}.json`
  override `Config` defaults. Cross-check the JSON fields against the
  inline defaults in each trainer's `Config` (cited at the top of
  each setup flowchart) to confirm the pilot uses the recipe you
  expect.
- **Catching code-vs-doc drift.** §3.6 / §4.6 / §5.7 list every drift
  found at extraction time. New drifts will appear if a trainer is
  edited without updating its `pipeline_docs/*.md`.

**Source extraction:** `git rev-parse HEAD` at notebook creation =
the working tree state at 2026-05-01 (post-rename commit `450c0b1`).
Re-run the extraction agents in `/home/silas/.claude/plans/vast-bouncing-treehouse.md`
after a non-trivial trainer edit.
